# Unicorn Time-to-Scale

**Question:** what predicts how fast a company reaches a one billion dollar valuation, and does the apparent acceleration over time survive scrutiny?

This notebook follows the build order in `unicorn-project-plan.md`. Cleaning decisions are documented in markdown cells as they are made, not retrofitted later.

## 1. First look

Before writing any cleaning logic, we look at the raw data as it actually is: shape, dtypes, and the first few rows. The project plan already tells us what to expect (negative durations, an old outlier, currency strings), but the point of this step is to verify that against the data itself rather than trust the summary.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/raw/unicorn_companies.csv')
df.shape

(1074, 10)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Company           1074 non-null   object
 1   Valuation         1074 non-null   object
 2   Date Joined       1074 non-null   object
 3   Industry          1074 non-null   object
 4   City              1058 non-null   object
 5   Country/Region    1074 non-null   object
 6   Continent         1074 non-null   object
 7   Year Founded      1074 non-null   int64 
 8   Funding           1074 non-null   object
 9   Select Investors  1073 non-null   object
dtypes: int64(1), object(9)
memory usage: 84.0+ KB


In [34]:
df.head()

,Company,Valuation,Date Joined,Industry,City,Country/Region,Continent,Year Founded,Funding,Select Investors
0,Bytedance,$180B,4/7/17,Artificial intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$100B,12/1/12,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100B,7/3/18,E-commerce & direct-to-consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95B,1/23/14,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG"
4,Klarna,$46B,12/12/11,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita..."


In [35]:
df.describe()

,Year Founded
count,1074.000000
mean,2012.895717
std,5.698573
min,1919.000000
25%,2011.000000
50%,2014.000000
75%,2016.000000
max,2021.000000


In [36]:
df.duplicated().sum()

np.int64(0)

In [37]:
df['Company'].duplicated().sum()

np.int64(1)

In [38]:
df[df['Company'].duplicated(keep=False)].sort_values('Company')

,Company,Valuation,Date Joined,Industry,City,Country/Region,Continent,Year Founded,Funding,Select Investors
40,Bolt,$11B,5/29/18,Auto & transportation,Tallinn,Estonia,Europe,2013,$1B,"Didi Chuxing, Diamler, TMT Investments"
44,Bolt,$11B,10/8/21,Fintech,San Francisco,United States,North America,2014,$1B,"Activant Capital, Tribe Capital, General Atlantic"


## Checking for other values

In [39]:
df['Valuation'].str[-1].unique()

array(['B'], dtype=object)

In [40]:
df['Funding'].str[-1].unique()

array(['B', 'M', 'n'], dtype=object)

In [41]:
df[df['Funding'].str[-1] == 'n']['Funding'].unique()

array(['Unknown'], dtype=object)

In [42]:
df['Funding'].isna().sum()

np.int64(0)

In [52]:
def parse_currency(value):
    if value == 'Unknown' or value == "unknown":
        return np.nan

    numeric_part = value.replace('$', '')

    if numeric_part.endswith('B'):
        return float(numeric_part[:-1]) * 1e9
    elif numeric_part.endswith('M'):
        return float(numeric_part[:-1]) * 1e6

    return np.nan

df['valuation_numeric'] = df['Valuation'].apply(parse_currency)
df['funding_numeric'] = df['Funding'].apply(parse_currency)

In [53]:
df[['Valuation', 'valuation_numeric']].head()

,Valuation,valuation_numeric
0,$180B,1.800000e+11
1,$100B,1.000000e+11
2,$100B,1.000000e+11
3,$95B,9.500000e+10
4,$46B,4.600000e+10


In [54]:
df[['Funding', 'funding_numeric']].sample(10)

,Funding,funding_numeric
588,$226M,2.260000e+08
416,$218M,2.180000e+08
700,$384M,3.840000e+08
517,$345M,3.450000e+08
1036,$326M,3.260000e+08
883,$538M,5.380000e+08
566,$517M,5.170000e+08
248,$1B,1.000000e+09
532,$450M,4.500000e+08
265,$274M,2.740000e+08


In [55]:
df['funding_numeric'].isna().sum()

np.int64(12)

In [56]:
(df['Funding'] == 'Unknown').sum()

np.int64(12)

In [58]:
df['date_joined_parsed'] = pd.to_datetime(df['Date Joined'], format='%m/%d/%y')

In [59]:
df['date_joined_parsed'].dt.year.min(), df['date_joined_parsed'].dt.year.max()

(2007, 2022)

In [60]:
df['years_to_unicorn'] = df['date_joined_parsed'].dt.year - df['Year Founded']

In [61]:
df['years_to_unicorn'].describe()

count    1074.000000
mean        7.000931
std         5.329672
min        -4.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        98.000000
Name: years_to_unicorn, dtype: float64

In [62]:
df[df['years_to_unicorn'] < 0][['Company', 'Date Joined', 'Year Founded', 'years_to_unicorn']]

,Company,Date Joined,Year Founded,years_to_unicorn
714,Yidian Zixun,10/17/17,2021,-4


In [63]:
df = df[df['years_to_unicorn'] >= 0].copy()
df.shape

(1073, 14)

In [75]:
df[df['Year Founded'] < 1990][['Company', 'Year Founded', 'Date Joined', 'years_to_unicorn']].sort_values('Year Founded')

,Company,Year Founded,Date Joined,years_to_unicorn
189,Otto Bock HealthCare,1919,6/24/17,98
373,Promasidor Holdings,1979,11/8/16,37
699,Five Star Business Finance,1984,3/26/21,37


In [76]:
df[df['Year Founded'] >= 1990]['Year Founded'].min()

1990

In [77]:
df = df[df['Year Founded'] >= 1990].copy()
df.shape

(1070, 14)

In [78]:
sorted(df['Country/Region'].unique())

['Argentina',
 'Australia',
 'Austria',
 'Bahamas',
 'Belgium',
 'Bermuda',
 'Brazil',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Croatia',
 'Czech Republic',
 'Denmark',
 'Estonia',
 'Finland',
 'France',
 'Germany',
 'Hong Kong',
 'India',
 'Indonesia',
 'Ireland',
 'Israel',
 'Italy',
 'Japan',
 'Lithuania',
 'Luxembourg',
 'Malaysia',
 'Mexico',
 'Netherlands',
 'Nigeria',
 'Norway',
 'Philippines',
 'Senegal',
 'Singapore',
 'South Africa',
 'South Korea',
 'Spain',
 'Sweden',
 'Switzerland',
 'Thailand',
 'Turkey',
 'United Arab Emirates',
 'United Kingdom',
 'United States',
 'Vietnam']